# 2.6 — Auditing the tagging, and closing Task C

Part I asks whether the tagging mechanisms earn their keep: is coreference actually
resolving to the right company, and does the anaphora recency heuristic contribute
anything once coref exists? It is the measurement `config.py` cites for
`USE_ANAPHORA_FALLBACK = False`, and it took three attempts — the first two were invalid,
for two different reasons that are worth reading before designing any similar audit.

Part II corrects Part I. The headline coref figure here was measured on span rows only,
which flattered the channel; Part II widens the sample, measures both populations
separately, and blends them by their true corpus weights.

The coref channel's accuracy is stated in Part II §2–§3 and nowhere else; the labelling
conventions behind those figures changed afterwards, and notebook 2.7 §12 records what
that moved.


---

# Part I — Which tagging mechanisms earn their keep?


### 0. Setup

In [1]:
import numpy as np
import pandas as pd
from stock_predictor.config import DATA_DIR, PROJ_ROOT

pd.set_option("display.max_colwidth", 140)
sentences = pd.read_parquet(DATA_DIR / "sentences.parquet")
body = sentences[~sentences["is_boilerplate"]]
print("non-boilerplate sentences:", len(body))
print("target sentences:", int(body["mentions_target"].sum()))

2026-08-16 22:03:31.234 | INFO     | stock_predictor.config:<module>:12 - PROJ_ROOT path is: D:\ML\stock-predictor


non-boilerplate sentences: 57250
target sentences: 13669


## 1. What is being audited

Three mechanisms tag a sentence without the word "Tesla" appearing in it:

| mechanism | how it decides |
|---|---|
| **coreference** | a neural model reads the whole article and links mention chains |
| **anaphora heuristic** | "the company" means whichever company was named most recently, within 6 sentences |
| **CEO alias** | the sentence names Musk; flagged as CEO-related but deliberately NOT tagged as Tesla |

They are mutually exclusive and ordered: an explicit name wins, then coref, then the heuristic as a
fallback. So the heuristic only ever runs on sentences coref could not handle.

## 2. Attempt one, and why it was worthless

The first audit showed an auditor each sentence **on its own** and asked whether it was about Tesla.
It returned coref 0.59, anaphora 0.21, and I nearly reported those as precision.

They are not precision. **Coref and the heuristic resolve pronouns using article context, and I had
removed the context.** For exactly the sentences these mechanisms exist to handle -- "It's a wonderful
business", "the company also saw its first-ever decline in annual revenue" -- the auditor could not
possibly verify the referent, and my instructions told it to answer `no` when no company could be
identified. So "the algorithm was wrong" and "this sheet cannot tell" were collapsed into the same
number. Roughly half the failures were the second kind.

The auditor flagged this itself, unprompted:

> *"the reported `no` rate overstates true error rate and should be read as 'unverifiable from
> sentence alone,' not 'algorithm was wrong.'"*

**The lesson generalises beyond this notebook:** an evaluation must give the judge at least the
evidence the system had. Ours had a whole article; the judge had one sentence.

## 3. Attempt two: context, referent-first, and an explicit ambiguous option

Three changes:

1. **Context.** Each row carries the 5 preceding sentences and 1 following -- matching
   `ANAPHORA_MAX_GAP = 6`, the window the heuristic itself uses.
2. **Ask for the referent, do not ask for a grade.** The first sheet asked "is our claim correct?",
   which invites agreement. The second asks *"what does this phrase refer to?"* as free text and never
   shows our answer. We compare afterwards.
3. **`AMBIGUOUS` is a first-class verdict.** If a careful reader *with* context still cannot tell,
   the confident tag is unjustified either way -- that is a finding about the mechanism, not a hole in
   the audit. Collapsing it into "wrong" is what destroyed attempt one.

To keep (2) honest the substitution preview had the injected company masked to `<NAME>`, so judging
the rewrite could not leak the referent.

> **Superseded by Part II.** The coref accuracy in this section — 93.5%, against a 26.4%
> no-span share — was measured on span rows only and overstates the channel. Part II
> re-measures it on both populations and is where the channel's accuracy is stated. The
> method and the error taxonomy here are unaffected.

### 3.1 The result, and my second mistake

Attempt two gave **coref 74%, anaphora 13%**, with the auditor finding the passage determinate in
**99%** of rows -- so unlike attempt one these were real error rates, not artifacts.

But the substitution numbers were wrong, and again it was my bug. My mask replaced `"Tesla's"` with
`"<NAME>"`, **eating the possessive**. A perfectly correct substitution --

    "Tesla's revenue growth of 25.52% is notably higher..."

was shown to the auditor as

    "<NAME> revenue growth of 25.52% is notably higher..."

which reads as a dropped possessive, and was duly marked broken. 40 of the 85 reported substitution
failures were this artifact. The tell was in the auditor's own report: *"There is not a single
`<NAME>'s` anywhere in the sheet"* -- which should have been impossible if the pipeline were producing
possessives at all, and it was.

Corrected, substitution accuracy was 0.789 rather than 0.60. The referent findings were untouched,
since the mask only ever altered the substitution column.

## 4. What we changed

Three changes, each tied to a measurement.

**Anaphora heuristic switched off** (`USE_ANAPHORA_FALLBACK = False`). 13% correct against coref's
74%, on a mechanism that only fires where coref already failed. The failures are exactly what a
recency rule produces -- 61 of them resolved to a *different company named in the same passage*:

> "**Pilot** is the largest network of travel centers in North America... **The company** serves an
> average of 1.2 million guests per day." → tagged Tesla

> "Ellison owns 41% of **Oracle**... Ellison owned 22% of **the company** 15 years ago" → tagged Tesla

Kept behind a flag rather than deleted, so it can be restored if coref coverage ever regresses.

**First-person plural removed from substitutable anaphora.** `we`/`our`/`us` in news text are nearly
always inside a quote from a *person*: *"'We want the future to look like the future,' Musk said"*
became *"Tesla want the future to look like the future"*.

**Contractions expanded rather than split.** The span covers `It` while the text reads `It's`, so
replacing the span orphaned the clitic: `"It's also profitable"` → `"Tesla's also profitable"` (which
means something different), and `"we're making big investments"` → `"Tesla're making..."`. A clitic
glued to a bare pronoun is now consumed and expanded -- `It's` → `Tesla is`.

In [2]:
# Regenerating with these changes moved the corpus as follows.
deltas = pd.DataFrame({
    "before": [14746, 23213, 4030, 1333, 2465],
    "after": [14433, 22193, 4030, 0, 1925],
}, index=["mentions_target", "mentions_other", "resolved_by_coref",
          "resolved_by_anaphora", "substitutions"])
deltas["delta"] = deltas["after"] - deltas["before"]
deltas

,before,after,delta
mentions_target,14746,14433,-313
mentions_other,23213,22193,-1020
resolved_by_coref,4030,4030,0
resolved_by_anaphora,1333,0,-1333
substitutions,2465,1925,-540


`resolved_by_coref` is **unchanged at 4,030**, which is the check that matters: switching off the
fallback did not disturb the mechanism we kept. `mentions_other` falls by more than `mentions_target`
because the heuristic resolved rival companies too.

## 5. Attempt three: the clean audit

200 coref-resolved sentences from the fixed pipeline, context window intact, referent asked first,
and the mask now mapping `"Tesla's"` → `"<NAME>'s"` so grammar survives and only identity is hidden.
The sheet was asserted to contain `<NAME>'s` rows before being sent -- the previous run's zero was the
bug.

In [3]:
key = pd.read_parquet(PROJ_ROOT / "references" / "context-audit-key-v2.parquet") \
    if (PROJ_ROOT / "references" / "context-audit-key-v2.parquet").exists() else None
lab = pd.read_csv(PROJ_ROOT / "references" / "context-audit-labels-v2.csv")
sheet = pd.read_csv(PROJ_ROOT / "references" / "context-audit-sheet-v2.csv")
d = sheet[["sample_id", "highlighted_phrase"]].merge(lab, on="sample_id")
# pandas reads TRUE/FALSE as booleans and NA as NaN, so normalise via str.
d["ref"] = d["refers_to_tesla"].astype(str).str.upper()
d["span"] = d["substitution_span_ok"].astype(str).str.upper()
d["has_decision"] = d["highlighted_phrase"].notna() & (d["highlighted_phrase"].astype(str) != "")

print(f"rows audited: {len(d)}")
print(f"  carrying a resolution decision: {int(d['has_decision'].sum())}")
print(f"  no phrase resolved            : {int((~d['has_decision']).sum())}")

rows audited: 200
  carrying a resolution decision: 138
  no phrase resolved            : 62


**A sampling flaw the auditor caught.** 62 of the 200 rows have no highlighted phrase at all -- they
are coref-*tagged* but nothing substitutable was found, so there is no resolution decision to judge.
I sampled on the tag rather than on the presence of a span. Those rows are excluded from the rates
below; they are not model errors, and counting them as such would repeat exactly the mistake of
attempt one.

In [4]:
real = d[d["has_decision"]]
print("=== REFERENT (rows carrying a decision) ===")
print(real["ref"].value_counts().to_string())
print(f"precision: {(real['ref'] == 'TRUE').mean():.3f}")
print()
sub = real[real["span"].isin(["TRUE", "FALSE"])]
print(f"=== SUBSTITUTION: {(sub['span'] == 'TRUE').mean():.3f} sound  (n={len(sub)}) ===")
print()
bad = real[(real["ref"] != "TRUE") | (real["span"] == "FALSE")]
print("failure classes:")
print(bad["note"].astype(str).str.lower().value_counts().to_string())

=== REFERENT (rows carrying a decision) ===
ref
TRUE         129
FALSE          8
AMBIGUOUS      1
precision: 0.935

=== SUBSTITUTION: 0.957 sound  (n=138) ===

failure classes:
note
refers to a different company                3
refers to a non-company antecedent           3
plural referent - refers to two companies    2
subject-verb disagreement                    1
antecedent outside context window            1


> **Superseded by Part II.** The coref accuracy in this section — 93.5%, against a 26.4%
> no-span share — was measured on span rows only and overstates the channel. Part II
> re-measures it on both populations and is where the channel's accuracy is stated. The
> method and the error taxonomy here are unaffected.

**Coreference: 93.5% correct referent, 95.7% mechanically sound substitutions.** Only 8 referent
errors in 138 decisions, and 5 of the 6 substitution failures are downstream of a referent error --
just one broke while the referent was right.

The residual errors are a short list, and none has a cheap fix:

- **3 resolved to a different company** (Figure AI, Tractor Supply, SpaceX)
- **3 resolved to a non-company antecedent** -- a percentage, an ETF, a trading alert. *"Tesla's US
  market share dropped to 45%. In 2019, **it** was 80%"* → the antecedent is the figure, and the
  substitution reads *"In 2019, Tesla was 80%."*
- **2 plural referents** collapsed to one company -- *"The problem for **both those companies** was
  **their** expanding capex"* became *"...was Tesla's expanding capex"*
- **1 antecedent above the context window**, the only genuine `AMBIGUOUS`

The auditor found the passage determinate in **137 of 138** rows, so context depth is essentially
never the binding constraint. Whatever headroom remains is in the resolver.

In [5]:
cor = body[body["mentions_target"] & body["resolved_by_coref"]]
print("coref-tagged target sentences:", len(cor))
print(f"  with a usable span (substituted)  : {int(cor['anaphor_char_start'].notna().sum())} "
      f"({100*cor['anaphor_char_start'].notna().mean():.1f}%)")
print(f"  no span (scored unsubstituted)    : {int(cor['anaphor_char_start'].isna().sum())} "
      f"({100*cor['anaphor_char_start'].isna().mean():.1f}%)")

coref-tagged target sentences: 2617
  with a usable span (substituted)  : 1925 (73.6%)
  no span (scored unsubstituted)    : 692 (26.4%)


**The honest caveat on the headline.** 26.4% of coref-tagged sentences carry no usable span, so the
93.5% describes the three quarters that do. The remaining quarter are still tagged as Tesla sentences
and still scored -- on unchanged text, with no aspect anchor -- and this audit says nothing about
whether those tags are right. Some of that quarter exists *because* of the fixes above: a sentence
whose only coref mention was "we" now correctly yields no span, which suppresses the bad substitution
while leaving the tag in place.

That is the next thing to measure, and it is a different question from the one asked here.

## 6. Where this leaves the pipeline

| mechanism | verdict | evidence |
|---|---|---|
| explicit name | keep | 11,052 sentences, a lookup rather than a judgement |
| **coreference (span rows)** | **keep** | 93.5% referent, 95.7% substitution, 2,617 sentences |
| **coreference (no-span rows)** | **keep, flagged noisy** | 56.5% referent (95% CI [49.0%, 63.7%], n=170) -- see notebook 2.7 |
| **coreference (blended, population-weighted)** | -- | **78.3%**, not 93.5% -- see notebook 2.7 |
| **anaphora heuristic** | **off** | 13% referent; cost 313 target sentences (2.1%) |
| CEO alias | unchanged | never sets `mentions_target`; unaudited here |

Final corpus: **13,669 non-boilerplate target sentences** — 11,052 explicit, 2,617 coref, 0 heuristic.

### Limitations

- **The 93.5% below is a SPAN-ONLY figure, not a channel figure.** It describes the rows carrying a
  decision at the time this notebook was written. Notebook 2.7 closes that gap: the no-span quarter
  (originally 26.4% of coref tags, measured at 1,189 rows / 23.5% of the current corpus) runs at
  56.5% correct (n=170), and the population-weighted blend across both populations is **78.3%**.
  Any statement of "coref accuracy" outside this notebook should cite the 78.3% blended figure, not
  93.5%.
- **The auditor is an LLM**, so this is protocol-driven adjudication, not human validation. It is
  reproducible and it was blind to our answers, which is the most that is available cheaply.
- **"The stock" was counted as Tesla.** 13-15 rows resolve a phrase to Tesla's *shares* rather than
  the company. If a downstream feature ever needs those separated, the precision figure drops to
  roughly 83%.
- **Three rows needed world knowledge** rather than an in-window antecedent (a Q3 delivery report
  among mega-caps; "the EV maker... its 10 millionth vehicle"). A resolver without that knowledge
  would legitimately fail them.
- **Nothing here measures recall.** We audited what the pipeline claimed, never what it missed. A
  sentence that should have been tagged and was not is invisible to this design.

### On the method itself

Two of the three attempts were invalidated by my own design errors -- removing the context the system
had, then masking away the grammar the auditor was asked to judge. Both were caught by the auditor's
own report rather than by me, and in both cases the tell was a number that was *too extreme to be
plausible* (67% of everything wrong; not one possessive in 213 substitutions). That is the practical
heuristic worth keeping: when an evaluation says almost everything is broken, suspect the evaluation
first.

---

# Part II — Closing Task C: what Part I should have said

*(formerly notebook 2.7 §1–§7)*

Part I reported 93.5% coref referent accuracy. That figure was measured on rows where
coref produced a **substitutable span**, which is not the whole channel: roughly a third
of coref-tagged sentences carry no usable span, and those are much worse. This part
widens the no-span sample, measures both populations, and blends them by their true
corpus weights.

**This is the authoritative statement of coref accuracy.**


## 1. The widened sample

In [1]:
import numpy as np
import pandas as pd

eval_df = pd.read_parquet("../../data/eval/coref_eval_labelled.parquet")
print(f"Total labelled rows: {len(eval_df)}")
print(eval_df.groupby("has_span").size().rename("n"))
eval_df.head(3)

Total labelled rows: 270
has_span
False    170
True     100
Name: n, dtype: int64


,row_id,article_id,sent_idx,headline,ticker,text,absa_text,anaphor_char_start,anaphor_char_end,has_span,verdict,referent,note,borderline
0,SPAN-1,140759177,3,TSLA Stock Under Pressure: Analyst Says $600 B...,TSLA,"Gary Black, managing director of The Future Fu...","Gary Black, managing director of The Future Fu...",62,74,True,target,the target company,,False
1,SPAN-2,139592998,21,"Rivian Stock Is Cheap, but Does That Make It a...",TSLA,And while it may take a few years to achieve p...,And while it may take a few years to achieve p...,92,107,True,other,Rivian,,False
2,SPAN-3,137884563,8,Can Promises Alone Push Tesla to $2T in 2026? ...,TSLA,"You’re paying exactly 15 times forward sales, ...","You’re paying exactly 15 times forward sales, ...",83,93,True,target,the target company,,False


120 new no-span rows were sampled from the current coref no-span population (`resolved_by_coref
& anaphor_char_start.isna()` over `data/sentences.parquet`), excluding the article_id/sent_idx pairs
already in the original 150-row set, seed `20260818` (vs. the original `20260817`). Context window
is identical to the original build: headline + 4 preceding sentences + the flagged sentence + 1
following sentence. Each row was read in full context and judged `target` (the sentence genuinely
refers to Tesla) or `other` (it refers to something else -- a rival company, a person, an unrelated
topic, boilerplate), with a free-text `referent` and a `borderline` flag for genuinely defensible
either-way calls (mostly plural/dual-entity merger sentences -- "both firms", "Musk's companies").
Labelling for the 120 new rows was split across three parallel passes working from the same
instructions and worked examples; the convention -- borderline counts as `target`
unless the referent is genuinely plural/not-singularly-Tesla -- was held constant across all three.

## 2. Accuracy, before and after widening

In [2]:
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return center - half, center + half


span = eval_df[eval_df["has_span"]]
nospan = eval_df[~eval_df["has_span"]]

span_k, span_n = int((span["verdict"] == "target").sum()), len(span)
nospan_k, nospan_n = int((nospan["verdict"] == "target").sum()), len(nospan)

rows = [
    ("span (rewritten)", span_k, span_n),
    ("no-span (tagged only) -- OLD n=50", 26, 50),
    ("no-span (tagged only) -- NEW n=170", nospan_k, nospan_n),
]
for label, k, n in rows:
    lo, hi = wilson_ci(k, n)
    print(f"{label:38s} {k:3d}/{n:3d} = {k/n:6.1%}   95% CI [{lo:.1%}, {hi:.1%}]")

span (rewritten)                        90/100 =  90.0%   95% CI [82.6%, 94.5%]
no-span (tagged only) -- OLD n=50       26/ 50 =  52.0%   95% CI [38.5%, 65.2%]
no-span (tagged only) -- NEW n=170     117/170 =  68.8%   95% CI [61.5%, 75.3%]


| population | n | correct | 95% CI |
|---|---|---|---|
| span (rewritten) | 100 | 89.0% | [81.4%, 93.7%] |
| no-span, OLD sample | 50 | 52.0% | [38.5%, 65.2%] |
| **no-span, WIDENED sample** | **170** | **56.5%** | **[49.0%, 63.7%]** |

**The headline finding survives widening: no-span rows are running well below span rows, and the
interval is now tight enough to say so with confidence.** The point estimate moved from 52.0% to
56.5% -- inside the old CI, so nothing here contradicts the earlier read -- and the CI width dropped
from ±13.4pp to ±7.4pp. "Coin-flip accuracy" was a fair characterisation at n=50 (the interval
reached down to 38.5%); at n=170 the honest description is **"materially worse than the span
channel, running in the mid-50s, and not a literal coin flip"** -- the lower bound of the CI (49.0%)
sits just under 50%, so "not statistically distinguishable from chance" is defensible but "is a coin
flip" overstates it slightly in the pessimistic direction.

The two samples' point estimates differ by 4.5pp, comfortably inside the old sample's own margin of
error -- there is no evidence the two samples are measuring different populations, which is the
sanity check widening is supposed to pass before its number is trusted.

## 3. The blended figure — what notebook 2.6 should have said

In [3]:
# Current population sizes, recomputed from the live corpus (data/sentences.parquet), not the
# handoff's earlier snapshot -- these happen to be unchanged this session (corpus not regenerated).
sentences = pd.read_parquet("../../data/sentences.parquet")
coref = sentences[sentences["resolved_by_coref"]]
span_pop = int(coref["anaphor_char_start"].notna().sum())
nospan_pop = int(coref["anaphor_char_start"].isna().sum())
target_pop = int((sentences["mentions_target"] & ~sentences["is_boilerplate"]).sum())

span_acc = span_k / span_n
nospan_acc = nospan_k / nospan_n
blended = (span_pop * span_acc + nospan_pop * nospan_acc) / (span_pop + nospan_pop)

print(f"coref-resolved sentences   : {len(coref):6d}")
print(f"  with span (rewritten)    : {span_pop:6d}  ({span_pop/len(coref):.1%})")
print(f"  no span (tagged only)    : {nospan_pop:6d}  ({nospan_pop/len(coref):.1%})")
print(f"coref share of target set  : {len(coref)/target_pop:.1%}")
print()
print(f"span accuracy              : {span_acc:.1%}")
print(f"no-span accuracy           : {nospan_acc:.1%}")
print(f"BLENDED (population-weighted): {blended:.1%}")

coref-resolved sentences   :   3617
  with span (rewritten)    :   2428  (67.1%)
  no span (tagged only)    :   1189  (32.9%)
coref share of target set  : 24.8%

span accuracy              : 90.0%
no-span accuracy           : 68.8%
BLENDED (population-weighted): 83.0%


**The blended, population-weighted accuracy of the coref channel is 78.3%, not 93.5%.**

That number is what notebook 2.6's headline should have reported and what any future notebook or
report should cite as "coref accuracy" -- not 93.5%, which describes span rows only. Notebook 2.6
has been amended in place (its title cell and closing table now carry this correction and point
here) rather than left to silently mislead.

In absolute terms: no-span sentences are **32.9% of the whole coref-resolved population**
(coref itself is 24.8% of the whole non-boilerplate target set) -- and
**~8.1% of ALL non-boilerplate target sentences** in the corpus. That is not a rounding error --
roughly 1 in 12 target sentences in the training data is running through a channel correct only
56.5% of the time.

## 4. Where the no-span errors go — the referent tail

In [4]:
errors = eval_df[eval_df["verdict"] == "other"]
nospan_errors = errors[~errors["has_span"]]

print(f"Total errors across the full 270-row set : {len(errors)}")
print(f"  distinct referents                      : {errors['referent'].nunique()}")
print(f"No-span errors                            : {len(nospan_errors)}")
print(f"  distinct referents                      : {nospan_errors['referent'].nunique()}")
print()
errors["referent"].value_counts().head(12)

Total errors across the full 270-row set : 63
  distinct referents                      : 46
No-span errors                            : 53
  distinct referents                      : 43



referent
SpaceX                                7
BYD                                   5
Rivian                                3
Nova (Kimbal Musk's drone company)    3
Slate Auto                            2
SpaceX and xAI                        2
WeRide                                1
Unitree                               1
OpenAI                                1
Chancellor McCormick / LinkedIn       1
Agtonomy / DBL Partners               1
Uber                                  1
Name: count, dtype: int64

This is the same shape of finding the original n=50 sample surfaced, now on more than five times
the data: **68 distinct referents across 85 errors.** The distribution has a short head (SpaceX 7,
BYD 5, Rivian 3, Nova -- Kimbal Musk's drone company -- 3) and a long tail of one-off referents:
Agtonomy, Lightship, Unitree, WeRide, ChargePoint, Slate Auto, OpenAI, Uber, YMAG (an ETF), BOTT (a
robotics ETF), a Delaware court filing, Musk's personal net worth, a SAE technical-standard
definition, Reliance Jio, Xiaomi, Li Auto. **This remains the decisive evidence against any curated
roster of confusor companies** -- no list of a practical size reaches that tail, and per the
project's ticker-agnostic constraint the tail would look entirely different for another ticker.
Framing the problem as referent verification — asking what a sentence is about, rather than
enumerating what it might be confused with — is the reading the data supports.

## 5. New failure modes surfaced by the wider sample

The widened sample did not just narrow a confidence interval -- it surfaced failure shapes the
original n=50 sample was too small to show. None of these were visible in the earlier 52% headline.

**Third-party fund/ETF articles.** Sentences from articles about a fund that merely *holds* Tesla
as one basket constituent (YMAG, BOTT, Tuttle's "MAGO" Magnificent Seven ETF, Direxion's Defined
Income Boost suite) resolve to the fund, not Tesla. These are systematically mis-tagged and
sentiment-irrelevant to the target company specifically.

**Musk-family companies that are not Tesla.** Kimbal Musk's drone company Nova is the largest single
new referent (3 occurrences) -- a first-person-plural "we" in a multi-company interview is a strong
Tesla signal in an earnings-call context and an actively misleading one in an interview about a
sibling Musk venture. The pipeline has no way to distinguish the two contexts from sentence-local
evidence.

**Multi-topic headlines.** Video-transcript-style articles bundling several companies in one
headline ("Tesla European sales, Lucid Q4 earnings, Lamborghini: EV latest") produce sentences
entirely about the OTHER named company (Lucid's capital-raise plans) that still carry a Tesla tag
because Tesla appears in the headline. Worth checking as a class if this pattern recurs.

**Generic technical/legal definitions.** A sentence stating a regulatory or technical definition
(the SAE Level 2 autonomy standard) rather than describing Tesla specifically.

**Sentence-splitting/extraction fragments.** `NOSPAN-107` is a bare `"Li Auto Inc. (NASDAQ: LI)"`
fragment -- the sentence splitter produced a company-name-only fragment from what should have been
one sentence, and it resolved to Tesla. Different bug class from the referent-resolution errors
above (this one is upstream, in sentence boundary detection), noted for completeness.

**A hyphen-boundary alias miss (not an error, but worth recording).** `NOSPAN-53` and `NOSPAN-87`
both contain the literal hyphenated form `"Tesla-SpaceX"` / `"SpaceX-Tesla"`, which
`entity_filter._compile_alias_pattern()`'s boundary regex does not match as an explicit "Tesla"
mention (a hyphen blocks the word-boundary check by design, to stop URL-slug false positives like
`tsla-stock-analysis`). Both rows are genuinely about Tesla, so this did not produce a labelling
error, but it explains why these otherwise-explicit mentions fell through to the coref/no-span path
at all. Not fixed here -- flagging for whoever next touches `_compile_alias_pattern()`.

## 6. Boilerplate — a finding that changed on closer inspection

In [5]:
empty_absa = eval_df[eval_df["absa_text"].fillna("") == ""]
check = empty_absa.merge(
    sentences[["article_id", "sent_idx", "is_boilerplate"]],
    on=["article_id", "sent_idx"],
    how="left",
)
check[["row_id", "has_span", "is_boilerplate", "text"]]

,row_id,has_span,is_boilerplate,text
0,SPAN-65,True,True,The company has multiple vehicles in its fleet...
1,NOSPAN-31,False,True,Any views or opinions expressed may not reflec...
2,NOSPAN-38,False,True,Want the latest recommendations from Zacks Inv...
3,NOSPAN-44,False,True,Read the complete narrative.
4,NOSPAN-82,False,True,Contact Zacks Investment Research 800-767-3771...
5,NOSPAN-131,False,True,"Get stock recommendations, portfolio guidance,..."
6,NOSPAN-162,False,True,Get the latest stock analysis from Benzinga?


Two open items had been recorded separately: "4 of the 24
no-span errors are promotional boilerplate that `is_boilerplate` failed to catch" and a standalone
data-quality bug, `SPAN-65` has a recorded span but an empty `absa_text`, not yet investigated.

**Both readings were wrong, and the correction is good news.** Checking the actual `is_boilerplate`
column (not available in the labelling context sheet, so the labelling passes could only guess from
the sentence text looking like boilerplate) shows `is_boilerplate` is **already `True`** for every
one of these seven rows, including `SPAN-65`. `needs_score()` -- the single predicate shared by
`score_sentence_table()` and `aggregate_article_features()` -- correctly excludes all of them from
FinBERT and ABSA scoring, which is exactly why `absa_text` is empty: **that is correct behaviour,
not a bug.** `SPAN-65`'s text repeats across exactly 5 distinct articles (the `BOILERPLATE_MIN_ARTICLES`
threshold), confirming it. Both mis-diagnoses came from judging boilerplate-ness by eye without checking the column that actually
decides it -- a small methodological lesson worth keeping: **check the flag, don't infer it from the
text.** The eval parquet's `note` field on these seven rows has been corrected in place.

This closes the `SPAN-65` investigation listed as outstanding and removes the
false "boilerplate detection is failing" finding from the record. `is_boilerplate` mis-tags
`mentions_target=True` onto these sentences (a coref artifact -- the sentence still gets a company
tag before boilerplate flagging runs, since `flag_boilerplate()` is a separate, later pass), but
that tag is inert: nothing downstream reads a boilerplate row's score.

## 7. Summary — Task C is now closed

| item | status |
|---|---|
| Widen no-span sample beyond n=50 | Done — n=170, CI tightened from ±13.4pp to ±7.4pp |
| Correct Part I's headline | Done — Part I now points here |
| Investigate `SPAN-65` empty-`absa_text` | Done — not a bug; `is_boilerplate` correctly excludes it (§6) |

**The number to carry forward: the coref channel's blended, population-weighted accuracy is 78.3%**
(89.0% on the 67.1%-share span population, 56.5% on the 32.9%-share no-span population). The
no-span channel is the pipeline's weakest measured link, now backed by n=170 instead of n=50, and
its error tail — 68 distinct referents across 85 errors in the full 270-row set — is the strongest
evidence against solving this with a curated list of confusable companies.

> **These three figures moved afterwards.** Two labelling conventions changed once the eval set was
> re-examined, taking it from 85 errors to 63: a company's own products, joint referents including
> the target, and funds holding it all count as the target, while inverse instruments do not. Under
> the current conventions the channel reads **90.0% span, 68.8% no-span, 83.1% blended**. Nothing
> about the pipeline changed — only what a label means. Notebook 2.7 §12 has the before-and-after,
> and `fusion.py`'s `PROVENANCE_CHANNELS` comment carries the current triple.

The channel was now measured, and measuring it fixed nothing. The programme that tried to repair it
is notebook 2.7.


---

## What happened next

Task C closed with the channel measured but not repaired. The four-stage programme that
followed — ensemble disagreement, an eval harness, the provenance split, and a local LLM
verification judge — is **notebook 2.7**, which now contains only that programme.

| notebook | question |
|---|---|
| `2.0` | What does the pipeline do? |
| `2.4` | Which fused sentence score ships? |
| **this one** | Do the tagging mechanisms earn their keep, and how accurate is coref really? |
| `2.7` | Can the coref channel be *repaired*? (Stages 4/4b/4c, 3, 2, 1) and §13, the magnitude weighting |
